In [9]:
mol_type = "6r7m"
n_mol = 3
T = 3.1
x = "Vector{Float64}([])"
comment = ""
bnds = 150.0
rs = 1.4
η = 0.3665
#σ_r = 0.2
#σ_t = 1.5
overlap_jump = 0.0
#overlap_slope = 1.1
delaunay_eps = 100.0

comment = replace(comment, " " => "_")
simulation_time_minutes = 24.0 * 60.0
input_specifier = "cc_rwm_ma_$(n_mol)_$(mol_type)"

"cc_rwm_ma_3_6r7m"

In [10]:
simulations_per_combination = 25

open("../configs/$(input_specifier)_config.txt", "w") do io
    i = 1
    println(io,"ArrayTaskID input_string")
    output_directory = "../Simulations/unsorted_output/$(input_specifier)/"
    for σ_t in [1.25, 1.5]
        for σ_r in [0.25, 0.3]
            for overlap_slope in [1.1]
                for _ in 0:simulations_per_combination-1
                    name = "$(i)_$(input_specifier)"
                    input_string = escape_string("name=\"$name\";x=$(x);T=$(T);rs=$rs;η=$η;σ_t=$σ_t;σ_r=$σ_r;overlap_jump=$overlap_jump;overlap_slope=$overlap_slope;bnds=$bnds;n_mol=$n_mol;mol_type=\"$mol_type\";simulation_time_minutes=$simulation_time_minutes;output_directory=\"$output_directory\";delaunay_eps=$delaunay_eps;comment=\"$comment\"")
                    println(io, "$i $input_string")
                    i += 1
                end
            end
        end
    end
end

In [11]:
total_simulations = length(readlines("../configs/$(input_specifier)_config.txt")) - 1

100

In [12]:
hours = Int(round(simulation_time_minutes / 60.0))
days = hours ÷ 24
remaining_hours = hours % 24
remaining_hours_string = remaining_hours < 10 ? "0$(remaining_hours)" : string(remaining_hours)
buffer_time_string = simulation_time_minutes < 5 ? "0$(Int(simulation_time_minutes)+2)" : "30"

open("../$(input_specifier)_script.job", "w") do io
    println(io, "#!/bin/bash")
    println(io, "#SBATCH --job-name=SolvationSimulations")
    println(io, "#SBATCH --time=0$(days)-$(remaining_hours_string):$(buffer_time_string)")
    println(io, "#SBATCH --ntasks=1")
    println(io, "#SBATCH --cpus-per-task=1")
    println(io, "#SBATCH --array=1-$(total_simulations)")
    println(io, "#SBATCH --chdir=/work/spirandelli/MorphoMolHPC/")
    println(io, "#SBATCH -o ./job_log/$(input_specifier)/%a.out # STDOUT")
    println(io, "")
    println(io, "export http_proxy=http://proxy2.uni-potsdam.de:3128 #Setting proxy, due to lack of Internet on compute nodes.")
    println(io, "export https_proxy=http://proxy2.uni-potsdam.de:3128")
    println(io, "")
    println(io, "module purge")
    println(io, "module load lang/Julia/1.7.3-linux-x86_64")
    println(io, "")
    println(io, "# Specify the path to the config file")
    println(io, "config=hpc_scripts/configs/$(input_specifier)_config.txt")
    println(io, "")
    println(io, "# Extract the variables from config file")
    println(io, "config_string=\$(awk -v ArrayTaskID=\$SLURM_ARRAY_TASK_ID '\$1==ArrayTaskID {print \$2}' \$config)")
    println(io, "")
    println(io, "julia -e \"$(escape_string("include(\"julia_scripts/cc_rwm_ma_call.jl\"); cc_rwm_ma_call(\"\$config_string\")"))\"")
end